In [1]:
# https://github.com/2manoj1/g-colab/blob/main/deepseek_AI_Agent.ipynb

In [9]:
!uv pip install -U langchain-deepseek

Resolved 34 packages in 259ms                                        
Prepared 1 package in 13ms                                               
Installed 1 package in 13ms1.0.0                            
 + langchain-deepseek==1.0.0


In [ ]:
!export OPENAI_API_KEY="sk-REDACTED-SET-VIA-ENVIRONMENT"
!export OPENAI_BASE_URL="https://api.openai.com/v1"

In [12]:
from __future__ import annotations
import os
import json
from dataclasses import dataclass
from typing import Dict, List, Optional, Annotated
from typing_extensions import TypedDict


import pandas as pd
import numpy as np
import yfinance as yf


# LangChain (modern imports)
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.messages import HumanMessage, AIMessage

In [ ]:
os.environ["OPENAI_API_KEY"] = "sk-REDACTED-SET-VIA-ENVIRONMENT"
os.environ["OPENAI_BASE_URL"] = "https://api.openai.com/v1"

In [13]:
ChatDeepSeek = None # type: ignore
_HAS_DEEPSEEK = False

from langchain_openai import ChatOpenAI
# LangGraph
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages

In [28]:
# ----------------------------------------------------------------------------------
# LLM setup (keeps spirit of original: DeepSeek via OpenAI-compatible or native)
# ----------------------------------------------------------------------------------
def setup_llm(model: Optional[str] = None, temperature: float = 0.2):
    if _HAS_DEEPSEEK and os.getenv("DEEPSEEK_API_KEY"):
        return ChatDeepSeek(model=model or os.getenv("DEEPSEEK_MODEL", "deepseek-chat"),
        temperature=temperature)
        
    # Otherwise use OpenAI-compatible client (works with DeepSeek via base_url)
    api_key = os.getenv("OPENAI_API_KEY") or os.getenv("DEEPSEEK_API_KEY")
    base_url = os.getenv("OPENAI_BASE_URL") or os.getenv("DEEPSEEK_BASE_URL")
    # Default to DeepSeek public endpoint if only DEEPSEEK_API_KEY exists
    
    print("API Key:", "set" if api_key else "not set")
    print("Base URL:", base_url or "not set")
    if base_url is None and os.getenv("DEEPSEEK_API_KEY"):
        base_url = "https://api.deepseek.com"
        
    return ChatOpenAI(model=model or os.getenv("OPENAI_MODEL", "gpt-4o-mini"),
                        temperature=temperature,
                        api_key=api_key,
                        base_url=base_url
    )

In [15]:
def rsi(series: pd.Series, period: int = 14) -> pd.Series:
    delta = series.diff()
    gain = delta.clip(lower=0)
    loss = -delta.clip(upper=0)
    avg_gain = gain.rolling(period, min_periods=period).mean()
    avg_loss = loss.rolling(period, min_periods=period).mean()
    rs = avg_gain / avg_loss.replace(0, np.nan)
    return 100 - (100 / (1 + rs))

In [16]:
TECH_PROMPT = PromptTemplate.from_template(
    """You are a technical analyst. Using the following indicators for {symbol},
    write a concise analysis (<120 words) and mention trend, momentum, and risks.


    DATA:
    last_close: {last_close}
    SMA20: {sma20}
    SMA50: {sma50}
    SMA200: {sma200}
    RSI14: {rsi14}
    1D%: {ret1d}
    5D%: {ret5d}
    1M%: {ret1m}
    """
)

In [17]:
MKT_PROMPT = PromptTemplate.from_template(
    """You are a market analyst. Using the metadata and peers for {symbol},
    summarize sector/industry context, valuation, and beta risk (<120 words).


    META:
    {meta}


    PEERS:
    {peers}
    """
    )

In [18]:
NEWS_PROMPT = PromptTemplate.from_template(
    """You are a news analyst. Read these recent headlines for {symbol} and write
    a short sentiment & key-themes summary (<100 words). If empty, say so.
    HEADLINES:
    {headlines}
    """
    )

In [19]:
RECO_PROMPT = PromptTemplate.from_template(
    """Synthesize a final recommendation for {symbol} combining TECH, MARKET, NEWS.
    Be balanced and specific (<120 words). Return a short paragraph.
    TECH: {tech}
    MARKET: {market}
    NEWS: {news}
    """
    )

In [20]:
# ----------------------------------------------------------------------------------
# LangGraph State (keeps original keys: messages, symbol, llm, results)
# ----------------------------------------------------------------------------------
class State(TypedDict):
    messages: Annotated[list, add_messages]
    symbol: str
    llm: object
    results: Dict[str, str]

In [21]:
# ----------------------------------------------------------------------------------
# Node: Technical Analysis
# ----------------------------------------------------------------------------------
def node_technical(state: State) -> State:
    symbol = state["symbol"].upper().strip()
    df = yf.Ticker(symbol).history(period="6mo", interval="1d")
    if df.empty or "Close" not in df:
        tech = f"No historical data for {symbol}."
    else:
        close = df["Close"].dropna()
        last = close.iloc[-1]
        sma20 = close.rolling(20).mean().iloc[-1] if len(close) >= 20 else np.nan
        sma50 = close.rolling(50).mean().iloc[-1] if len(close) >= 50 else np.nan
        sma200 = close.rolling(200).mean().iloc[-1] if len(close) >= 200 else np.nan
        rsi14 = rsi(close, 14).iloc[-1] if len(close) >= 14 else np.nan
        def pct(n):
            return None if len(close) <= n else float((close.iloc[-1] / close.iloc[-(n+1)] - 1) * 100)
        chain = TECH_PROMPT | state["llm"] | StrOutputParser()
        tech = chain.invoke({
            "symbol": symbol,
            "last_close": f"{float(last):.2f}",
            "sma20": "na" if pd.isna(sma20) else f"{float(sma20):.2f}",
            "sma50": "na" if pd.isna(sma50) else f"{float(sma50):.2f}",
            "sma200": "na" if pd.isna(sma200) else f"{float(sma200):.2f}",
            "rsi14": "na" if pd.isna(rsi14) else f"{float(rsi14):.2f}",
            "ret1d": pct(1),
            "ret5d": pct(5),
            "ret1m": pct(21)
        })
    state["results"]["technical"] = tech
    state["messages"].append(AIMessage(content=f"[TECH]{tech}"))
    return state

In [22]:
# ----------------------------------------------------------------------------------
# Node: Market Analysis
# ----------------------------------------------------------------------------------
def node_market(state: State) -> State:
    symbol = state["symbol"].upper().strip()
    tk = yf.Ticker(symbol)
    info = tk.info or {}
    
    meta = {
        "sector": info.get("sector"),
        "industry": info.get("industry"),
        "market_cap": info.get("marketCap"),
        "beta": info.get("beta"),
        "pe": info.get("trailingPE") or info.get("forwardPE"),
    }
    
    peers = []
    chain = MKT_PROMPT | state["llm"] | StrOutputParser()
    market = chain.invoke({"symbol": symbol, "meta": json.dumps(meta, default=str), "peers": peers})
    state["results"]["market"] = market
    state["messages"].append(AIMessage(content=f"[MARKET]{market}"))
    return state

In [23]:
# ----------------------------------------------------------------------------------
# Node: News Analysis
# ----------------------------------------------------------------------------------
def node_news(state: State) -> State:
    symbol = state["symbol"].upper().strip()
    tk = yf.Ticker(symbol)
    headlines = []
    try:
        news = getattr(tk, "news", None)
        if news:
            for n in news[:8]:
                t = n.get("title") or n.get("content") or ""
                src = n.get("publisher") or n.get("source") or ""
        if t:
            headlines.append(f"- {t} ({src})")
    except Exception:
        pass
    text = "\n".join(headlines) if headlines else "(no recent headlines found)"
    chain = NEWS_PROMPT | state["llm"] | StrOutputParser()
    news_out = chain.invoke({"symbol": symbol, "headlines": text})
    state["results"]["news"] = news_out
    state["messages"].append(AIMessage(content=f"[NEWS]\n{news_out}"))
    return state

In [24]:
# ----------------------------------------------------------------------------------
# Node: Final Recommendation
# ----------------------------------------------------------------------------------
def node_recommendation(state: State) -> State:
    symbol = state["symbol"].upper().strip()
    chain = RECO_PROMPT | state["llm"] | StrOutputParser()
    reco = chain.invoke({
        "symbol": symbol,
        "tech": state["results"].get("technical", ""),
        "market": state["results"].get("market", ""),
        "news": state["results"].get("news", ""),
    })
    state["results"]["recommendation"] = reco
    state["messages"].append(AIMessage(content=f"[RECO]{reco}"))
    return state

In [25]:
# ----------------------------------------------------------------------------------
# Build Graph
# ----------------------------------------------------------------------------------
def create_analysis_graph():
    graph = StateGraph(State)
    graph.add_node("technical", node_technical)
    graph.add_node("market", node_market)
    graph.add_node("news", node_news)
    graph.add_node("recommendation", node_recommendation)


    graph.add_edge(START, "technical")
    graph.add_edge("technical", "market")
    graph.add_edge("market", "news")
    graph.add_edge("news", "recommendation")
    graph.add_edge("recommendation", END)


    return graph.compile()

In [29]:
# ----------------------------------------------------------------------------------
# Public API
# ----------------------------------------------------------------------------------
def run_analysis(symbol: str, model: Optional[str] = None, temperature: float = 0.2) -> Dict[str, str]:
    llm = setup_llm(model=model, temperature=temperature)
    app = create_analysis_graph()
    init: State = {
    "messages": [HumanMessage(content=f"Analyze {symbol}")],
    "symbol": symbol,
    "llm": llm,
    "results": {}
    }
    state = app.invoke(init)
    return state["results"]

In [32]:
run_analysis("AAPL", model="gpt-4o-mini")

API Key: set
Base URL: https://api.openai.com/v1


{'technical': 'AAPL is currently in an upward trend, closing at 268.81, significantly above its 20-day SMA of 255.75 and 50-day SMA of 244.89, indicating strong bullish momentum. The RSI of 62.25 suggests that the stock is approaching overbought territory, which could signal a potential pullback. The recent performance shows a 1D gain of 2.28%, a 5D gain of 2.51%, and a 1M gain of 5.23%, reflecting positive short-term momentum. However, risks include potential overextension and market volatility, which could lead to corrections. Investors should monitor these indicators closely for any signs of trend reversal.',
 'market': "Apple Inc. (AAPL) operates within the Technology sector, specifically in the Consumer Electronics industry. With a market capitalization of approximately $3.99 trillion, AAPL is a dominant player in this space. The company's price-to-earnings (P/E) ratio stands at 40.79, indicating a premium valuation relative to earnings, reflecting strong investor confidence and g